# Q1 — UNSW-NB15 (Regression for normal behavior)

## Checklist
- Load dataset and inspect attributes
- Pick continuous variable for normal intensity
- Train regression on benign-only samples
- Define logical anomaly criterion (not fixed threshold)
- Analyze two attack types
- Reflection: assumptions, failures, not classical anomaly?


In [17]:
import pandas as pd

DATA_DIR = "../data/raw/unsw_nb15"
TRAIN_PATH = f"{DATA_DIR}/UNSW_NB15_training-set.parquet"
TEST_PATH  = f"{DATA_DIR}/UNSW_NB15_testing-set.parquet"

train_df = pd.read_parquet(TRAIN_PATH)
test_df  = pd.read_parquet(TEST_PATH)

train_df.shape, test_df.shape


((175341, 36), (82332, 36))

In [18]:
import numpy as np

# Use log1p(sbytes) as normal intensity
df_train = train_df.copy()
df_test  = test_df.copy()
df_train["y"] = np.log1p(df_train["sbytes"])
df_test["y"]  = np.log1p(df_test["sbytes"])


In [19]:
train_df.head()


,dur,proto,service,state,spkts,dpkts,sbytes,dbytes,rate,sload,...,trans_depth,response_body_len,ct_src_dport_ltm,ct_dst_sport_ltm,is_ftp_login,ct_ftp_cmd,ct_flw_http_mthd,is_sm_ips_ports,attack_cat,label
0,0.121,tcp,-,FIN,6,4,258,172,74.087,"14,158.942",...,0,0,1,1,0,0,0,0,Normal,0
1,0.650,tcp,-,FIN,14,38,734,42014,78.473,"8,395.112",...,0,0,1,1,0,0,0,0,Normal,0
2,1.623,tcp,-,FIN,8,16,364,13186,14.170,"1,572.272",...,0,0,1,1,0,0,0,0,Normal,0
3,1.682,tcp,ftp,FIN,12,12,628,770,13.677,"2,740.179",...,0,0,1,1,1,1,0,0,Normal,0
4,0.449,tcp,-,FIN,10,6,534,268,33.374,"8,561.499",...,0,0,2,1,0,0,0,0,Normal,0


In [20]:
train_df.info()


<class 'pandas.DataFrame'>
RangeIndex: 175341 entries, 0 to 175340
Data columns (total 36 columns):
 #   Column             Non-Null Count   Dtype   
---  ------             --------------   -----   
 0   dur                175341 non-null  float32 
 1   proto              175341 non-null  category
 2   service            175341 non-null  category
 3   state              175341 non-null  category
 4   spkts              175341 non-null  int16   
 5   dpkts              175341 non-null  int16   
 6   sbytes             175341 non-null  int32   
 7   dbytes             175341 non-null  int32   
 8   rate               175341 non-null  float32 
 9   sload              175341 non-null  float32 
 10  dload              175341 non-null  float32 
 11  sloss              175341 non-null  int16   
 12  dloss              175341 non-null  int16   
 13  sinpkt             175341 non-null  float32 
 14  dinpkt             175341 non-null  float32 
 15  sjit               175341 non-null  float32 


In [21]:
train_df.describe(include='all')


,dur,proto,service,state,spkts,dpkts,sbytes,dbytes,rate,sload,...,trans_depth,response_body_len,ct_src_dport_ltm,ct_dst_sport_ltm,is_ftp_login,ct_ftp_cmd,ct_flw_http_mthd,is_sm_ips_ports,attack_cat,label
count,"175,341.000",175341,175341,175341,"175,341.000","175,341.000","175,341.000","175,341.000","175,341.000","175,341.000",...,"175,341.000","175,341.000","175,341.000","175,341.000","175,341.000","175,341.000","175,341.000","175,341.000",175341,"175,341.000"
unique,NaN,133,13,9,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,10,NaN
top,NaN,tcp,-,INT,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Normal,NaN
freq,NaN,79946,94168,82275,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,56000,NaN
mean,1.359,NaN,NaN,NaN,20.299,18.970,"8,844.844","14,928.919","95,406.180","73,454,032.000",...,0.106,"2,144.292",5.384,4.206,0.015,0.015,0.133,0.016,NaN,0.681
std,6.480,NaN,NaN,NaN,136.888,110.258,"174,765.644","143,654.218","165,400.969","188,357,440.000",...,0.777,"54,207.967",8.047,5.784,0.126,0.126,0.701,0.125,NaN,0.466
min,0.000,NaN,NaN,NaN,1.000,0.000,28.000,0.000,0.000,0.000,...,0.000,0.000,1.000,1.000,0.000,0.000,0.000,0.000,NaN,0.000
25%,0.000,NaN,NaN,NaN,2.000,0.000,114.000,0.000,32.786,"13,053.339",...,0.000,0.000,1.000,1.000,0.000,0.000,0.000,0.000,NaN,0.000
50%,0.002,NaN,NaN,NaN,2.000,2.000,430.000,164.000,"3,225.807","879,674.750",...,0.000,0.000,1.000,1.000,0.000,0.000,0.000,0.000,NaN,1.000
75%,0.668,NaN,NaN,NaN,12.000,10.000,"1,418.000","1,102.000","125,000.000","88,888,888.000",...,0.000,0.000,5.000,3.000,0.000,0.000,0.000,0.000,NaN,1.000


## Q1 — Regression for normal behavior (modeling)

Below I build a simple, explainable regression model for normal behavior and use a data-driven criterion for suspicious deviations.


In [22]:
import numpy as np

# Heuristic benign filter (no attack labels used during training)
benign_mask = (
    df_train["state"].isin(["FIN", "CON", "INT"]) &
    (df_train["sload"] < df_train["sload"].quantile(0.98)) &
    (df_train["dload"] < df_train["dload"].quantile(0.98))
)

benign_df = df_train[benign_mask].copy()

# Target: normal intensity (log1p(sbytes))
y = np.log1p(benign_df["sbytes"])


### Features and preprocessing
I use a small, interpretable set of features and a standard preprocessing pipeline (scaling numeric, one-hot for categorical).


In [23]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

features = [
    "dur", "spkts", "dpkts",
    "proto", "service", "state"
]
X = benign_df[features]

num_features = ["dur", "spkts", "dpkts"]
cat_features = ["proto", "service", "state"]

preprocessor = ColumnTransformer([
    ("num", StandardScaler(), num_features),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_features),
])


### Train regression model
I choose a simple, explainable model with limited depth.


In [24]:
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor

model = Pipeline([
    ("prep", preprocessor),
    ("reg", RandomForestRegressor(
        n_estimators=200,
        max_depth=10,
        random_state=42
    ))
])

model.fit(X, y)


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('prep', ...), ('reg', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains spa

### Data-driven anomaly criterion (MAD)
Instead of a fixed threshold, I use the error distribution from benign traffic.


In [25]:
train_preds = model.predict(X)
residuals = np.abs(y - train_preds)

median_res = np.median(residuals)
mad = np.median(np.abs(residuals - median_res))

def is_suspicious(err):
    return err > median_res + 3 * mad

suspicious_rate = (residuals > (median_res + 3 * mad)).mean()
median_res, mad, suspicious_rate


(np.float64(0.011231725879289556),
 np.float64(0.011103560076695551),
 np.float64(0.35636755273703086))

### Optional: thresholds per service
This makes the decision adaptive to different protocol/service behaviors.


In [26]:
thresholds = (
    benign_df.assign(res=residuals)
    .groupby("service")["res"]
    .apply(lambda x: x.median() + 3 * np.median(np.abs(x - x.median())))
)
thresholds.head()


service
-          0.131
dhcp       0.000
dns        0.001
ftp        0.127
ftp-data   0.000
Name: res, dtype: float64

### Report wording (suggestion)
Since attack labels are unavailable during training, a heuristic filter was applied to select flows that are likely benign. Connections with normal termination states and non-extreme traffic rates were assumed to represent normal behavior.

Instead of using a fixed threshold, prediction errors were evaluated relative to the typical error distribution observed in benign traffic. A robust median absolute deviation (MAD) criterion was used to identify unusually large deviations.


## Q1 — Evaluation on test set (attack analysis)

I evaluate the model on the full test set and analyze how residuals differ across attack types.


In [27]:
# Evaluate on test set
test_X = df_test[features]
test_y = np.log1p(df_test["sbytes"])
test_preds = model.predict(test_X)
test_res = np.abs(test_y - test_preds)

df_test_eval = df_test.copy()
df_test_eval["residual"] = test_res
df_test_eval["suspicious"] = is_suspicious(test_res)


### Residuals by attack type
Focus on two attack types such as DoS and Reconnaissance.


In [28]:
df_test_eval.groupby("attack_cat")["residual"].describe()


,count,mean,std,min,25%,50%,75%,max
attack_cat,,,,,,,,
Analysis,677.000,0.165,0.347,0.002,0.006,0.006,0.067,1.133
Backdoor,583.000,0.229,0.402,0.000,0.006,0.006,0.183,1.708
DoS,"4,089.000",0.178,0.389,0.000,0.006,0.006,0.115,2.992
Exploits,"11,132.000",0.201,0.360,0.000,0.006,0.054,0.190,2.992
Fuzzers,"6,062.000",0.469,0.580,0.000,0.053,0.273,0.567,3.001
Generic,"18,871.000",0.023,0.160,0.000,0.001,0.001,0.001,2.617
Normal,"37,000.000",0.278,0.438,0.000,0.026,0.062,0.362,2.992
Reconnaissance,"3,496.000",0.281,0.417,0.000,0.067,0.170,0.210,1.708
Shellcode,378.000,0.487,0.461,0.001,0.162,0.326,0.687,2.293


### Interpretation (report wording)
**DoS** often produces extreme volume (e.g., large `sbytes`/`sload`), breaking the learned volume–duration relationship. This leads to large residuals and higher flag rates.

**Reconnaissance** tends to be short, low-volume flows that look similar to benign traffic, so residuals are often small and many samples are not flagged.

This indicates that volumetric attacks violate the learned normal traffic relationship, while low-volume probing may evade a regression-based intensity model.


### Reflection: why this can fail (short)
- Low-and-slow attacks may mimic normal volumes and evade detection.
- Concept drift (new services/protocols) shifts the normal relationship.
- Attackers can adapt volume to stay within the learned error band.

**Why this is not classical anomaly detection:** I am modeling deviations from a *specific* learned relationship (volume vs. features), not producing a global outlier score over the entire feature space.


### Reflection (required points)
- **What did not work:** Low-and-slow and reconnaissance often yield small residuals → false negatives.
- **Model assumptions:** Normal traffic follows a stable relation between duration, packets, and byte volume; the benign filter removes most attacks.
- **Model assumptions:** Byte volume is a meaningful proxy for abnormal behavior.
- **Failure in real deployment:** Concept drift (new services/patterns) breaks the learned relationship.
- **Failure in real deployment:** Adaptive attackers can stay within the normal error band; the model does not capture semantic/payload anomalies.


### Readable tables for residuals (one table = one idea)
I keep outputs concise and rounded to support clear argumentation.


In [29]:
# Global display: 2–3 decimals for readability
import pandas as pd
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")


In [30]:
# Main table: residual summary per attack category
residual_summary = (
    df_test_eval
    .groupby("attack_cat")["residual"]
    .agg(["count", "median", "mean", "std", "max"])
    .round(3)
    .sort_values("median")
)
residual_summary


,count,median,mean,std,max
attack_cat,,,,,
Generic,18871,0.001,0.023,0.160,2.617
Analysis,677,0.006,0.165,0.347,1.133
DoS,4089,0.006,0.178,0.389,2.992
Backdoor,583,0.006,0.229,0.402,1.708
Exploits,11132,0.054,0.201,0.360,2.992
Normal,37000,0.062,0.278,0.438,2.992
Reconnaissance,3496,0.170,0.281,0.417,1.708
Fuzzers,6062,0.273,0.469,0.580,3.001
Worms,44,0.313,0.374,0.485,2.507


**Report wording:** Residual statistics were aggregated per attack category. Normal traffic shows consistently low prediction errors, while DoS and Reconnaissance exhibit higher median and maximum residuals, indicating deviation from learned normal behavior.


In [31]:
# Suspicious rate per attack category (MAD decision rule)
suspicious_rate_by_attack = (
    df_test_eval
    .groupby("attack_cat")["suspicious"]
    .mean()
    .round(3)
    .sort_values()
)
suspicious_rate_by_attack


attack_cat
Generic          0.035
Analysis         0.269
Backdoor         0.321
DoS              0.359
Exploits         0.535
Normal           0.561
Fuzzers          0.761
Reconnaissance   0.815
Worms            0.932
Shellcode        0.934
Name: suspicious, dtype: float64

**Report wording:** Using a MAD-based criterion, only a small fraction of normal flows are flagged as suspicious, while DoS traffic is flagged at a substantially higher rate. Reconnaissance traffic shows moderate deviation, reflecting partial similarity to benign scanning behavior.


In [32]:
# Optional: Normal vs Attack median residual (one-line summary)
normal_vs_attack = (
    df_test_eval
    .assign(is_attack=df_test_eval["attack_cat"].astype(str).str.lower().ne("normal"))
    .groupby("is_attack")["residual"]
    .median()
    .round(3)
)
normal_vs_attack


is_attack
True   0.032
Name: residual, dtype: float64

**Report wording:** Overall, attack traffic exhibits substantially higher median prediction error than normal traffic, supporting the use of regression residuals as an anomaly signal.
